# ANOOS DevSecOps Pipeline — hero banner

Generates the repo social-preview banner with FLUX.1-dev on a free Kaggle T4.

**Why a diffusion model here and not for the diagrams:** flowcharts need deterministic,
legible text — that is Mermaid's job. The banner is art direction, which is what a
diffusion model is actually good at.

Runtime → Change runtime type → GPU T4 x2 → Run all.

In [ ]:
!pip install -q diffusers==0.31.0 transformers==4.46.2 accelerate==1.1.1 safetensors sentencepiece protobuf

In [ ]:
import torch
from diffusers import FluxPipeline

pipe = FluxPipeline.from_pretrained(
    'black-forest-labs/FLUX.1-dev',
    torch_dtype=torch.bfloat16,
)
pipe.enable_model_cpu_offload()
print('pipeline loaded on', pipe.device)

In [ ]:
PROMPT = (
    'wide cinematic banner for a devops engineering repository, dark navy background, '
    'isometric cutaway of three server nodes arranged left to right connected by glowing '
    'cyan pipelines, a forge anvil silhouette subtly embossed in the centre, '
    'abstract gears and container cubes floating, deep teal and amber rim lighting, '
    'volumetric haze, ultra clean composition, generous empty space on the right for text, '
    'no text, no letters, no words, high detail, 8k render'
)
NEGATIVE = 'text, letters, words, watermark, logo, blurry, low quality, cluttered, people'

image = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE,
    height=640,
    width=1536,
    num_inference_steps=28,
    guidance_scale=3.5,
    generator=torch.Generator('cpu').manual_seed(20261028),
).images[0]

image.save('/kaggle/working/banner.png')
image

In [ ]:
# Social preview needs 1280x640 exactly.
from PIL import Image
img = Image.open('/kaggle/working/banner.png').resize((1280, 640), Image.LANCZOS)
img.save('/kaggle/working/social-preview.png')
print(img.size)